In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa

def load_mono(path, sr=None):
    y, sr = librosa.load(path, sr=sr, mono=True)  # to mono, optional resample
    return y, sr

def make_same_length(a, b, mode="min"):
    if mode == "min":
        L = min(len(a), len(b))
        return a[:L], b[:L]
    elif mode == "max":
        L = max(len(a), len(b))
        a2 = np.pad(a, (0, L - len(a)))
        b2 = np.pad(b, (0, L - len(b)))
        return a2, b2
    else:
        raise ValueError("mode must be 'min' or 'max'")

def normalize_peak(y, peak=0.99):
    m = np.max(np.abs(y))
    return y if m == 0 else y * (peak / m)

def mix_two_sources_to_five(
    wav1, wav2, sr_target=None, out_dir=None, length_mode="min", add_noise_std=0.0
):
    # 1) Load
    s1, sr1 = load_mono(wav1, sr=sr_target)
    s2, sr2 = load_mono(wav2, sr=sr_target if sr_target is not None else sr1)

    # 2) Ensure same SR
    if sr_target is None and sr1 != sr2:
        # resample second to first if needed
        s2 = librosa.resample(s2, orig_sr=sr2, target_sr=sr1)
        sr = sr1
    else:
        sr = sr_target if sr_target is not None else sr1

    # 3) Match length
    s1, s2 = make_same_length(s1, s2, mode=length_mode)

    # 4) Stack sources: s has shape (2, T)
    s = np.vstack([s1[np.newaxis, :], s2[np.newaxis, :]])

    # 5) Choose/define a well-conditioned 5×2 mixing matrix A
    A = np.array([
        [0.9, 0.1],
        [0.7, 0.3],
        [0.5, 0.5],
        [0.3, 0.7],
        [0.1, 0.9],
    ], dtype=np.float32)

    x = A @ s

    # 7) Optional noise
    if add_noise_std > 0:
        x = x + np.random.randn(*x.shape) * add_noise_std

    # 8) Normalize per-mixture to avoid clipping, keep float32
    for i in range(x.shape[0]):
        x[i] = normalize_peak(x[i], peak=0.99)
    x = x.astype(np.float32)

    # 9) Save (optional)
    paths = []
    if out_dir is not None:
        out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
        for i in range(5):
            p = out / f"mix_{i+1}.wav"
            sf.write(p.as_posix(), x[i], sr, subtype="PCM_16")  # or 'FLOAT'
            paths.append(p.as_posix())

    return x, sr, A, paths

# ---------- Example usage ----------
# x: (5, T) mixtures, sr: sampling rate used, A: mixing matrix used

folder = "D:\Data_30_09_2025\Clipped(20m)"
aircraft_path = folder + "/A2-0002_OPT G_002_0001_Tr1.wav"
wind_path = "."

x, sr, A, paths = mix_two_sources_to_five(
    wav1="path/to/source1.wav",
    wav2="path/to/source2.wav",
    sr_target=22050,          # or None to keep native SR
    out_dir="synthetic_mixes",
    A=None,                   # or supply your own 5x2
    length_mode="min",        # truncate to the shorter source
    add_noise_std=0.0         # e.g., 0.005 for light noise
)

print("Mixtures shape:", x.shape)  # (5, T)
print("SR:", sr)
print("A:\n", A)
print("Saved:", paths)

